# Final results

Read saved metrics and plots from the original run or bundled results. This notebook does not train or evaluate the model. The final holdout has already been evaluated.

In [ ]:
import csv
from html import escape
import json

from IPython.display import HTML, display
from PIL import Image

from train import PROJECT_DIR, REPO_DIR

iteration_dir = PROJECT_DIR / "runs/iterations/data_growth_20261108"
if not (iteration_dir / "final_evaluation/metrics.json").exists():
    iteration_dir = REPO_DIR / "results/data_growth_20261108"
metrics = json.loads((iteration_dir / "final_evaluation/metrics.json").read_text())
config = json.loads((iteration_dir / "final_candidate/config.json").read_text())
split = json.loads((iteration_dir / "split_summary.json").read_text())
print("Training images:", split["split_counts"]["train"])
print("Validation images:", split["split_counts"]["validation"])
print("Final-test images:", metrics["count"])
print("Supported countries:", len(config["classes"]))
print("Results:", iteration_dir)
print("Packaged checkpoint:", REPO_DIR / "web_demo/model.pt")
print("Model SHA-256:", metrics["model_sha256"])

In [ ]:
results = [
    ("Accuracy", f"{metrics['accuracy']:.2%}"),
    ("Macro F1", f"{metrics['macro_f1']:.3f}"),
    ("Balanced accuracy", f"{metrics['balanced_accuracy']:.2%}"),
    ("Top-three accuracy", f"{metrics['top3_accuracy']:.2%}"),
    ("Correct predictions", f"{metrics['correct']} / {metrics['count']}"),
    ("Uniform chance baseline", f"{1 / len(config['classes']):.2%}"),
]
display(HTML("<table><tr><th>Final held-out metric</th><th>Result</th></tr>" + "".join("<tr><td>" + escape(name) + "</td><td>" + escape(value) + "</td></tr>" for name, value in results) + "</table>"))

In [ ]:
display(Image.open(iteration_dir / "final_candidate" / "training_curves.png"))

In [ ]:
display(Image.open(iteration_dir / "final_evaluation" / "confusion_matrix_normalized.png"))

In [ ]:
with (iteration_dir / "final_evaluation" / "per_country_metrics.csv").open(newline="", encoding="ascii") as stream:
    countries = list(csv.DictReader(stream))
headers = ["Country", "Test images", "Precision", "Recall", "F1"]
body = []
for row in countries:
    values = [row["country"], row["support"], f"{float(row['precision']):.3f}", f"{float(row['recall']):.3f}", f"{float(row['f1']):.3f}"]
    body.append("<tr>" + "".join("<td>" + escape(value) + "</td>" for value in values) + "</tr>")
display(HTML("<table><tr>" + "".join("<th>" + name + "</th>" for name in headers) + "</tr>" + "".join(body) + "</table>"))
print("The final holdout has 15 to 100 images per country. Grouping does not establish geographic independence.")

In [ ]:
print((REPO_DIR / "results/README.md").read_text(encoding="utf-8"))